In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from transformers import TFViTModel, AutoImageProcessor
import numpy as np

# Load preprocessed dataset
data = np.load("./assets/preprocessed_data/dataset.npz", allow_pickle=True)

# Extract training, validation, and test sets
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]
class_names = data["class_names"].tolist()

print(f"Dataset Loaded! Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Convert labels to one-hot encoding
num_classes = len(class_names)
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_val = tf.keras.utils.to_categorical(y_val, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

# Load ViT processor
vit_model_name = "google/vit-base-patch16-224"
vit_processor = AutoImageProcessor.from_pretrained(vit_model_name)

# Preprocessing function for both EfficientNet and ViT
def preprocess_image(image, label):
    # Convert to float32 and normalize (EfficientNet expects this)
    image = tf.image.convert_image_dtype(image, tf.float32)

    # Preprocess for ViT
    vit_inputs = vit_processor(images=image.numpy(), return_tensors="tf")["pixel_values"]
    
    return image, vit_inputs, label

# Convert dataset into TensorFlow dataset
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Apply preprocessing and batching
train_dataset = (
    train_dataset.map(lambda x, y: tf.py_function(preprocess_image, [x, y], [tf.float32, tf.float32, tf.float32]))
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset.map(lambda x, y: tf.py_function(preprocess_image, [x, y], [tf.float32, tf.float32, tf.float32]))
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset.map(lambda x, y: tf.py_function(preprocess_image, [x, y], [tf.float32, tf.float32, tf.float32]))
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

# Build model function
def build_model(num_classes):
    # Load EfficientNetB4
    base_model1 = keras.applications.EfficientNetB4(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
    base_model1.trainable = False

    # Load ViT
    vit_model = TFViTModel.from_pretrained(vit_model_name)

    # Define input
    inputs = keras.Input(shape=(224, 224, 3))

    # EfficientNet Feature Extraction
    x1 = base_model1(inputs)
    x1 = layers.GlobalAveragePooling2D()(x1)  # Shape: (batch, 1792)

    # ViT Feature Extraction
    def vit_features(image):
        inputs = vit_processor(images=image, return_tensors="tf")["pixel_values"]
        outputs = vit_model(inputs).last_hidden_state[:, 0, :]  # CLS token
        return outputs

    x2 = tf.keras.layers.Lambda(lambda img: tf.py_function(vit_features, [img], tf.float32))(inputs)
    x2 = tf.reshape(x2, (-1, 768))  # ViT's output is 768-dimensional

    # Merge both feature vectors
    x = layers.Concatenate()([x1, x2])
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    return model

# Build and compile model
model = build_model(num_classes)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["categorical_accuracy"]
)

# Train the model
model.fit(train_dataset, validation_data=val_dataset, epochs=10)


2025-03-05 16:43:39.264538: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-05 16:43:39.306868: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Dataset Loaded! Train: 5313, Val: 1525, Test: 752


2025-03-05 16:43:48.690113: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFViTModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing TFViTModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFViTModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFViTModel were not initialized from the PyTorch model and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10


It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.


InvalidArgumentError: Graph execution error:

Detected at node 'Equal_2' defined at (most recent call last):
    File "/home/kiwi/projects/assignments/six_phases/parkinsons-mri/.pixi/envs/default/lib/python3.10/runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/home/kiwi/projects/assignments/six_phases/parkinsons-mri/.pixi/envs/default/lib/python3.10/runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "/home/kiwi/projects/assignments/six_phases/parkinsons-mri/.pixi/envs/default/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "/home/kiwi/projects/assignments/six_phases/parkinsons-mri/.pixi/envs/default/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
      app.start()
    File "/home/kiwi/projects/assignments/six_phases/parkinsons-mri/.pixi/envs/default/lib/python3.10/site-packages/ipykernel/kernelapp.py",

In [3]:
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

y_train = np.argmax(y_train, axis=-1)
y_val = np.argmax(y_val, axis=-1)
y_test = np.argmax(y_test, axis=-1)



y_train shape: (5313, 3, 3)
y_val shape: (1525, 3, 3)
y_test shape: (752, 3, 3)


In [ ]:
import numpy as np

# Load preprocessed dataset
data = np.load("./assets/preprocessed_data/dataset.npz", allow_pickle=True)

# List stored arrays
print("Stored arrays:", data.files)

# Extract training, validation, and test sets
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]
class_names = data["class_names"].tolist()

print(f"Dataset Loaded! Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


In [ ]:
num_classes = len(np.unique(y_train))
print(f"Number of classes: {num_classes}")


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from transformers import AutoImageProcessor, TFViTModel

def build_model(num_classes):
    # EfficientNetB4 from Keras Applications
    base_model1 = keras.applications.EfficientNetB4(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
    base_model1.trainable = False

    # Hugging Face ViT Model
    vit_model_name = "google/vit-base-patch16-224"
    vit_extractor = AutoImageProcessor.from_pretrained(vit_model_name)
    vit_model = TFViTModel.from_pretrained(vit_model_name)

    # Input Layer
    inputs = keras.Input(shape=(224, 224, 3))

    # EfficientNet Feature Extraction
    x1 = base_model1(inputs)
    x1 = layers.GlobalAveragePooling2D()(x1)

    # ViT Feature Extraction (requires preprocessing)
    def vit_features(image):
        image = tf.image.convert_image_dtype(image, tf.float32)
        image_np = tf.expand_dims(image, axis=0)  # Add batch dimension
        inputs = vit_processor(images=image_np, return_tensors="tf")["pixel_values"]
        outputs = vit_model(inputs).last_hidden_state[:, 0, :]  # CLS token
        return outputs

    x2 = tf.keras.layers.Lambda(lambda img: tf.py_function(vit_features, [img], tf.float32))(inputs)

    x2 = tf.reshape(x2, (-1, 768))

    # Merge both feature vectors
    x = layers.Concatenate()([x1, x2])
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)  # Multi-class classification

    model = keras.Model(inputs, outputs)
    return model



In [ ]:
model = build_model(num_classes=4)  # Adjust based on dataset

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["categorical_accuracy"]
)

# Train model
model.fit(train_dataset, validation_data=val_dataset, epochs=50)


In [ ]:
model.summary()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = np.argmax(test_labels, axis=1)
y_pred = np.argmax(model.predict(test_dataset), axis=1)

print(classification_report(y_true, y_pred))


In [ ]:
# Unfreeze top layers
base_model1.trainable = True
base_model2.trainable = True

# Recompile and fine-tune
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["categorical_accuracy"]
)
model.fit(train_dataset, validation_data=val_dataset, epochs=20)


In [ ]:
model.save("parkinsons_mri_model.h5")
